# Предсказание октановых чисел (RON/MON) методами машинного обучения

## Содержание

1. **Введение** -- RON, MON и их значение в нефтехимии
2. **Загрузка и разведочный анализ данных (EDA)**
3. **Построение молекулярных дескрипторов с RDKit**
4. **Отбор признаков (Feature Selection)**
5. **Обучение моделей** -- базовые модели и CatBoost
6. **Логирование экспериментов с MLflow**
7. **Конфигурация пайплайна с Hydra**
8. **Подбор гиперпараметров с Optuna**
9. **Итоговый пайплайн и выводы**

## 1. Введение

**RON** (Research Octane Number) и **MON** (Motor Octane Number) -- ключевые характеристики топлива, определяющие его детонационную стойкость.

- **RON** измеряется при стандартных условиях (600 об/мин)
- **MON** измеряется при более жестких условиях (900 об/мин, предварительный подогрев)
- **OS** (Octane Sensitivity) = RON - MON

Задача: по структуре молекулы (SMILES) предсказать RON и MON, используя молекулярные дескрипторы.

## 2. Загрузка и разведочный анализ данных (EDA)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

df = pd.read_csv("data.csv", encoding="utf-8-sig")
print(f"Dataset: {df.shape[0]} compounds, {df.shape[1]} columns")
df.head(10)

In [ ]:
# Проверим пропуски
missing = df.isnull().sum()
print("Пропущенные значения:")
print(missing[missing > 0])

# Удалим строки без целевой переменной RON
df_clean = df.dropna(subset=["RON", "MON"]).copy()
print(f"\nПосле удаления пропусков: {len(df_clean)} соединений")

### Задание 1: Визуализация распределений

Постройте 3 графика в одном ряду (`plt.subplots(1, 3, ...)`):
1. Барплот количества соединений по `FuelClass`
2. Гистограмма распределения `RON` с вертикальной линией среднего
3. Гистограмма распределения `MON` с вертикальной линией среднего

In [ ]:
# YOUR CODE HERE



### Задание 2: RON vs MON

Постройте scatter plot RON vs MON, раскрашенный по `FuelClass`.
Добавьте диагональную линию RON = MON для наглядности.

In [ ]:
# YOUR CODE HERE



### Задание 3: Boxplot по классам

Постройте boxplot RON и MON по классам топлив (отсортируйте по медиане RON).

In [ ]:
# YOUR CODE HERE



In [ ]:
# Визуализация молекул из разных классов
samples = df_clean.groupby("FuelClass").first().reset_index()
mols = [Chem.MolFromSmiles(s) for s in samples["Smiles"]]
legends = [f"{row['Name']}\nRON={row['RON']:.0f}" for _, row in samples.iterrows()]

img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 250), legends=legends)
img

## 3. Построение молекулярных дескрипторов с RDKit

### Подходы к описанию молекул:

| Тип дескриптора | Описание | Пример |
|:---|:---|:---|
| **2D дескрипторы** | Физико-химические свойства | MolWt, LogP, TPSA, HBA/HBD |
| **Фингерпринты** | Бинарное представление субструктур | Morgan (ECFP), MACCS, RDKit FP |
| **3D дескрипторы** | Геометрия молекулы | Asphericity, Eccentricity |
| **Graph-based** | GNN на молекулярном графе | GCN, MPNN, AttentiveFP |

### Задание 4: Вычисление 2D дескрипторов

Реализуйте функцию `compute_rdkit_2d_descriptors(smiles_list)`, которая:
- Принимает список SMILES
- Для каждого SMILES создает `Chem.MolFromSmiles(smi)` и вычисляет дескрипторы через `Descriptors`
- Возвращает `pd.DataFrame`

Используйте дескрипторы: `MolWt`, `MolLogP`, `TPSA`, `NumHAcceptors`, `NumHDonors`, `NumRotatableBonds`,
`NumAromaticRings`, `NumAliphaticRings`, `RingCount`, `FractionCSP3`, `HeavyAtomCount` и другие на ваш выбор.

Подсказка: `getattr(Descriptors, name)` возвращает функцию-дескриптор.

In [ ]:
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs


def compute_rdkit_2d_descriptors(smiles_list: list[str]) -> pd.DataFrame:
    # YOUR CODE HERE
    pass



### Задание 5: Morgan Fingerprints (ECFP)

Реализуйте функцию `compute_morgan_fingerprints(smiles_list, radius=2, n_bits=1024)`, которая:
- Создает генератор: `rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)`
- Для каждого SMILES вычисляет фингерпринт: `gen.GetFingerprintAsNumPy(mol)`
- Возвращает `pd.DataFrame` с колонками `morgan_0`, `morgan_1`, ...

In [ ]:
def compute_morgan_fingerprints(
    smiles_list: list[str], radius: int = 2, n_bits: int = 1024
) -> pd.DataFrame:
    # YOUR CODE HERE
    pass



### Задание 6: MACCS Keys

Реализуйте функцию `compute_maccs_keys(smiles_list)`, которая:
- Вычисляет MACCS-фингерпринт: `rdMolDescriptors.GetMACCSKeysFingerprint(mol)`
- Конвертирует в numpy: `DataStructs.ConvertToNumpyArray(fp, arr)`
- Возвращает `pd.DataFrame` (167 бит)

In [ ]:
def compute_maccs_keys(smiles_list: list[str]) -> pd.DataFrame:
    # YOUR CODE HERE
    pass



### Задание 7: Вычислите дескрипторы и объедините

Вызовите все три функции для `df_clean["Smiles"]` и объедините результаты 2D + Morgan через `pd.concat`.

In [ ]:
smiles = df_clean["Smiles"].tolist()

# YOUR CODE HERE: вычислите desc_2d, desc_morgan, desc_maccs


# Объедините 2D + Morgan в X_all



In [ ]:
# Посмотрим на распределение нескольких ключевых дескрипторов
key_descriptors = ["MolWt", "MolLogP", "TPSA", "FractionCSP3", "NumAromaticRings", "RingCount"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, desc in zip(axes.flatten(), key_descriptors):
    ax.hist(desc_2d[desc].dropna(), bins=25, edgecolor="black", alpha=0.7)
    ax.set_title(desc)
    ax.set_xlabel("Value")

plt.suptitle("Распределение ключевых 2D дескрипторов", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Задание 8: Корреляция дескрипторов с целевыми переменными

Вычислите корреляцию Пирсона 2D-дескрипторов с RON и MON.
Постройте два горизонтальных барплота (`.plot.barh()`).

Подсказка: `desc_2d.corrwith(pd.Series(y_ron))`

In [ ]:
y_ron = df_clean["RON"].values
y_mon = df_clean["MON"].values

# YOUR CODE HERE



## 4. Отбор признаков (Feature Selection)

При большом числе дескрипторов (особенно фингерпринтов) необходимо:

1. **Удалить константные/низковариативные** признаки
2. **Удалить сильно коррелированные** признаки (мультиколлинеарность)
3. **Отобрать наиболее информативные** признаки (mutual information, importance)

### Задание 9: Реализуйте функции отбора признаков

Реализуйте три функции:

1. `remove_low_variance(X, threshold)` -- используйте `VarianceThreshold` из sklearn
2. `remove_correlated(X, threshold)` -- удалите один из пары признаков с корреляцией > threshold
3. `select_by_mutual_info(X, y, n_features)` -- отберите top-N по `mutual_info_regression`

In [ ]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression


def remove_low_variance(X: pd.DataFrame, threshold: float = 0.01) -> pd.DataFrame:
    # YOUR CODE HERE
    pass


def remove_correlated(X: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    # YOUR CODE HERE
    pass


def select_by_mutual_info(
    X: pd.DataFrame, y: np.ndarray, n_features: int = 50
) -> pd.DataFrame:
    # YOUR CODE HERE
    pass



### Задание 10: Примените пайплайн отбора признаков

Последовательно примените: `remove_low_variance` -> `remove_correlated` -> `select_by_mutual_info`.

In [ ]:
X_all_filled = X_all.fillna(0)

# YOUR CODE HERE: примените 3 шага отбора



In [ ]:
# Корреляционная матрица отобранных 2D дескрипторов
desc_2d_selected = [c for c in X_selected.columns if not c.startswith("morgan_")]
if len(desc_2d_selected) > 2:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        X_selected[desc_2d_selected].corr(),
        annot=True, fmt=".2f", cmap="RdBu_r", center=0,
        square=True, ax=ax
    )
    ax.set_title("Корреляционная матрица отобранных 2D дескрипторов")
    plt.tight_layout()
    plt.show()

## 5. Обучение моделей

Сравним несколько подходов:
- **Linear Regression** (baseline)
- **Random Forest**
- **Gradient Boosting**
- **CatBoost**

### Задание 11: Разделение данных и функция оценки

1. Разделите данные на train/test (`train_test_split`, `test_size=0.2`, `random_state=42`)
2. Примените `StandardScaler` к train и test
3. Реализуйте функцию `evaluate_model(model, X_train, X_test, y_train, y_test, name)`,
   которая обучает модель, предсказывает и возвращает dict с MAE, RMSE, R2

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor


def evaluate_model(model, X_train, X_test, y_train, y_test, name: str) -> dict:
    # YOUR CODE HERE
    pass


# YOUR CODE HERE: train_test_split + StandardScaler



### Задание 12: Обучите и сравните модели

Создайте словарь моделей и обучите каждую:
- Linear Regression
- Ridge (alpha=1)
- Random Forest (n_estimators=200, max_depth=10)
- Gradient Boosting (n_estimators=200, learning_rate=0.1, max_depth=5)
- CatBoost (iterations=500, learning_rate=0.05, depth=6, verbose=0)

Учтите, что линейные модели требуют масштабирования (`X_train_scaled`), а деревья -- нет.

In [ ]:
# YOUR CODE HERE



### Задание 13: Визуализация результатов

Постройте 3 графика:
1. Барплот MAE по моделям
2. Барплот R2 по моделям
3. Scatter Predicted vs True RON для лучшей модели

In [ ]:
# YOUR CODE HERE



### Задание 14: Feature Importance

Выведите и визуализируйте top-20 признаков по важности (`feature_importances_`) для CatBoost.

In [ ]:
# YOUR CODE HERE



## 6. Логирование экспериментов с MLflow

**MLflow** позволяет:
- Отслеживать параметры, метрики и артефакты экспериментов
- Сравнивать разные запуски
- Воспроизводить результаты

```
# Запуск UI после выполнения ячеек:
# mlflow ui --backend-store-uri mlruns
```

In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.catboost

# Настраиваем MLflow
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("RON_MON_prediction")

print("MLflow настроен. Experiment: RON_MON_prediction")

### Задание 15: Логирование в MLflow

Для каждой модели создайте `mlflow.start_run()` и залогируйте:
- Параметры: `mlflow.log_param("model_type", name)`, n_features, target
- Метрики: `mlflow.log_metric("mae", mae)`, rmse, r2
- Модель: `mlflow.sklearn.log_model(model, "model")` или `mlflow.catboost.log_model(model, "model")`

In [ ]:
# YOUR CODE HERE



## 7. Конфигурация пайплайна с Hydra

**Hydra** позволяет:
- Выносить все параметры в YAML-конфиги
- Запускать эксперименты с разными конфигурациями из CLI
- Делать multirun (sweep) по параметрам

Конфиг `configs/config.yaml` уже создан. Посмотрим, как его использовать:

In [ ]:
# Загрузка конфига Hydra в ноутбуке
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
import os

config_dir = os.path.join(os.getcwd(), "configs")

with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

In [ ]:
# Пример использования конфига
print(f"Target: {cfg.data.target}")
print(f"Test size: {cfg.data.test_size}")
print(f"Model: {cfg.model.name}")
print(f"CatBoost iterations: {cfg.model.catboost.iterations}")
print(f"Morgan radius: {cfg.descriptors.morgan_radius}")
print(f"Feature selection method: {cfg.feature_selection.method}")

print("\n--- Пайплайн можно запускать из CLI: ---")
print("python pipeline.py")
print("python pipeline.py data.target=MON model.name=random_forest")
print("python pipeline.py --multirun model.catboost.depth=4,6,8")

## 8. Подбор гиперпараметров с Optuna

**Optuna** -- фреймворк для автоматического подбора гиперпараметров:
- Алгоритм TPE (Tree-structured Parzen Estimator)
- Pruning неперспективных trial-ов
- Визуализация пространства поиска

### Задание 16: Optuna -- целевая функция

Реализуйте `objective(trial)` для оптимизации CatBoost:

Используйте `trial.suggest_*` для параметров:
- `iterations`: int, 100-1000
- `learning_rate`: float, 0.01-0.3, log scale
- `depth`: int, 3-10
- `l2_leaf_reg`: float, 0.01-10, log scale
- `min_data_in_leaf`: int, 1-30

Оцените через `cross_val_score` с `scoring="neg_mean_absolute_error"` и верните `-scores.mean()`.

Создайте study и запустите `study.optimize(objective, n_trials=30, timeout=120)`.

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    # YOUR CODE HERE
    pass


# YOUR CODE HERE: create study and optimize



In [ ]:
# Визуализация результатов Optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# История оптимизации
plot_optimization_history(study, ax=axes[0])
axes[0].set_title("История оптимизации")

# Важность гиперпараметров
plot_param_importances(study, ax=axes[1])
axes[1].set_title("Важность гиперпараметров")

plt.tight_layout()
plt.show()

### Задание 17: Финальная модель

1. Обучите CatBoost с лучшими параметрами из `study.best_params`
2. Выведите MAE, RMSE, R2 на тестовой выборке
3. Залогируйте результат в MLflow (run_name="CatBoost_Optuna_best")

In [ ]:
# YOUR CODE HERE



### Задание 18: Финальная визуализация

Постройте два графика:
1. Predicted vs True RON для лучшей модели
2. Анализ остатков (residuals vs predicted)

In [ ]:
# YOUR CODE HERE



## 9. Итоговый пайплайн

Файл `pipeline.py` объединяет все шаги в единый пайплайн с Hydra-конфигурацией:

```bash
# Базовый запуск
python pipeline.py

# Предсказание MON вместо RON
python pipeline.py data.target=MON

# Другая модель
python pipeline.py model.name=random_forest

# Sweep по гиперпараметрам
python pipeline.py --multirun model.catboost.depth=4,6,8 model.catboost.learning_rate=0.01,0.05,0.1

# Просмотр результатов в MLflow
mlflow ui --backend-store-uri mlruns
```

### Что мы изучили:

1. **RDKit** -- вычисление 2D дескрипторов, Morgan/MACCS fingerprints
2. **Feature Selection** -- VarianceThreshold, корреляционный фильтр, Mutual Information
3. **Модели** -- от линейных до gradient boosting (CatBoost)
4. **MLflow** -- логирование параметров, метрик, моделей
5. **Hydra** -- управление конфигурацией пайплайна
6. **Optuna** -- автоматический подбор гиперпараметров

### Задания для самостоятельной работы

1. Обучите модель для предсказания **MON** вместо RON. Сравните качество.
2. Добавьте **MACCS fingerprints** к признакам и оцените влияние на качество.
3. Попробуйте использовать только **структурные дескрипторы из датасета** (P, S, T, ...) без RDKit-дескрипторов.
4. Реализуйте **кросс-валидацию с группами** (GroupKFold по FuelClass) -- как изменится качество?
5. Настройте Optuna для оптимизации **одновременно RON и MON** (multi-objective).